<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Deep Learning & Neural Networks** · Week 2 · Training Deep Networks Well · Lesson 10 of 24

# Initialisation, Batch Norm and Vanishing Gradients

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

**This lesson uses a GPU.** Before you start: **Runtime → Change runtime type → T4 GPU → Save**. The code also runs without a GPU, only more slowly.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks).

**Learning objective:** Explain why gradients can vanish in deep networks, measure it, and fix it with ReLU, good initialisation, batch normalisation and residual connections.

## Why deep networks were hard to train

Until about 2010, networks with more than a few layers barely learned. The reason is in backpropagation: the gradient for an early layer is a **product** of many local slopes, one per layer. The sigmoid's slope is at most 0.25. Multiply ten numbers of 0.25 or less and you get almost nothing — the early layers receive no learning signal. This is the **vanishing gradient** problem. (If the slopes are bigger than 1, gradients can **explode** instead.)

## Measure it

We build a 10-layer network, run one backward pass, and print the average gradient size of each layer, from the first to the last:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits

digits = load_digits()
X = torch.tensor(digits.data[:200] / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target[:200])

def deep_net(activation, he_init=False, batch_norm=False, depth=10):
    torch.manual_seed(0)
    layers = []
    for i in range(depth):
        linear = nn.Linear(64, 64)
        if he_init:
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")
            nn.init.zeros_(linear.bias)
        layers.append(linear)
        if batch_norm:
            layers.append(nn.BatchNorm1d(64))
        layers.append(activation())
    layers.append(nn.Linear(64, 10))
    return nn.Sequential(*layers)

def gradient_sizes(model):
    F.cross_entropy(model(X), y).backward()
    return [f"{m.weight.grad.abs().mean().item():.0e}" for m in model if isinstance(m, nn.Linear)]

print("sigmoid:         ", gradient_sizes(deep_net(nn.Sigmoid)))
print("ReLU + He init:  ", gradient_sizes(deep_net(nn.ReLU, he_init=True)))
print("sigmoid + BN:    ", gradient_sizes(deep_net(nn.Sigmoid, batch_norm=True)))

What you should see:

- **Sigmoid:** the first layer's gradients are around **0.00000000001** (1e-11), while the last layer's are around 0.01 — a difference of about a billion times. The early layers cannot learn.
- **ReLU with He initialisation:** every layer's gradient is about the same size (around 1e-3). Healthy.
- **Sigmoid with batch normalisation:** also healthy, even with the sigmoid.

## Fix 1 — ReLU

ReLU's slope is exactly 1 for positive inputs, so it does not shrink the gradient as it passes through. That simple change is a big part of why "deep" learning took off.

## Fix 2 — Good initialisation

Weights should start at a size that keeps signals from shrinking or growing from layer to layer:

- **He (Kaiming) initialisation** for ReLU: random normal with variance 2 ÷ (number of inputs).
- **Xavier (Glorot) initialisation** for tanh/sigmoid: variance about 1 ÷ (number of inputs).

PyTorch's layers already use sensible defaults; for very deep ReLU networks, `nn.init.kaiming_normal_` makes it explicit. This is also why you scaled weights by √(2/64) in the Week 1 project.

## Fix 3 — Batch normalisation

**Batch norm** standardises each neuron's outputs over the current mini-batch (mean 0, standard deviation 1 — like StandardScaler, but inside the network), then lets the network learn its own scale and shift. It keeps every layer's inputs in a healthy range, allows bigger learning rates and often speeds training up a lot.

In [ ]:
bn = nn.BatchNorm1d(3)
batch = torch.tensor([[1.0, 200.0, -5.0], [3.0, 400.0, -1.0], [5.0, 600.0, 3.0]])
bn.train()
print(bn(batch).detach().round(decimals=2))

Each column comes out with mean 0 and similar spread, whatever its original scale. Like dropout, batch norm behaves differently in `train()` and `eval()` mode: at prediction time it uses averages remembered from training.

## Fix 4 — Residual connections

A **residual** (skip) connection adds a block's input to its output: output = x + block(x). The gradient can flow straight through the "+" — a highway past the layers. Residual networks (**ResNets**, 2015) made networks with 100+ layers trainable, and every transformer uses them too.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return F.relu(x + self.layers(x))      # the skip connection

block = ResidualBlock(64)
print(block(torch.randn(5, 64)).shape)

## Exploding gradients and clipping

When gradients explode (common in recurrent networks, Week 4), **gradient clipping** caps their size before the update:

In [ ]:
model = deep_net(nn.ReLU, he_init=True)
F.cross_entropy(model(X), y).backward()
total = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print("gradient size before clipping:", round(total.item(), 3))

Put it between `loss.backward()` and `optimizer.step()`.

## Key terms

- **Vanishing gradient** — gradients shrink towards zero in early layers, so they stop learning.
- **Exploding gradient** — gradients grow huge and make training unstable.
- **He / Xavier initialisation** — starting weight sizes that keep signals healthy.
- **Batch normalisation** — standardising layer outputs over each mini-batch.
- **Residual connection** — adding a block's input to its output.
- **Gradient clipping** — capping the gradient size.

**Takeaway:** Deep networks train well when gradients can flow: use ReLU, sensible initialisation, batch normalisation and skip connections, and clip gradients if they explode.

## Try it yourself

1. Repeat the measurement with `nn.Tanh`. Is it better or worse than sigmoid?
2. Use ReLU **without** He initialisation (the PyTorch default). How do the gradient sizes compare?
3. Train the 10-layer sigmoid network with and without batch norm on the digits for 20 epochs. Compare validation accuracy.
4. Stack four `ResidualBlock`s after an input layer and train them on the digits.

---
Finished? Go back to [Deep Learning & Neural Networks on WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*